# 🎬 Video Extraction & Analysis Tool

Analyze videos from YouTube, Vimeo, TikTok, direct URLs, or local files using **Claude AI vision** and **Whisper transcription**.

### Steps
1. Run cells 1–3 to install everything (~3 min)
2. Paste your Anthropic API key in cell 4
3. Run cell 5 to analyze any video URL


In [ ]:
# Cell 1 — Install system dependency (ffmpeg)
!apt-get install -y ffmpeg 2>&1 | tail -3
print('ffmpeg ready.')

In [ ]:
# Cell 2 — Clone repo and install Python dependencies
!git clone https://github.com/codedaddylive/claude-code-repo /content/video-tool 2>&1 | tail -3
%cd /content/video-tool
!pip install -r requirements.txt -q
print('All dependencies installed.')

In [ ]:
# Cell 3 — Run integration test (validates everything except Claude API)
import sys
sys.path.insert(0, '/content/video-tool')
!SKIP_WHISPER=0 python3 tests/integration_test.py

In [ ]:
# Cell 4 — Set your Anthropic API key
# Option A: type it directly (don't share this notebook after)
import os
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'  # <-- paste your key here

# Option B: use Colab Secrets (recommended)
# In the left sidebar click the key icon, add secret named ANTHROPIC_API_KEY
# then uncomment the two lines below:
# from google.colab import userdata
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

print('API key set:', 'yes' if os.environ.get('ANTHROPIC_API_KEY','').startswith('sk-') else 'NOT SET')

In [ ]:
# Cell 5 — Analyze a video
# Change VIDEO_URL to any YouTube, Vimeo, TikTok, or direct mp4 URL
VIDEO_URL = 'https://www.youtube.com/watch?v=BaW_jenozKc'  # ~1 min clip
MAX_FRAMES = 6
INTERVAL_SEC = 10

!python3 cli.py analyze "{VIDEO_URL}" \
    --max-frames {MAX_FRAMES} \
    --interval {INTERVAL_SEC} \
    --output /content/result.json

print('\nDone. Loading result...')
import json
with open('/content/result.json') as f:
    result = json.load(f)

print(f"\nDuration: {result.get('duration_sec', '?'):.1f}s")
print(f"Frames analysed: {result.get('frame_count', 0)}")
if result.get('visual_summary'):
    print(f"\n--- Visual Summary ---\n{result['visual_summary']}")
if result.get('transcription'):
    print(f"\n--- Transcript ---\n{result['transcription']['full_text'][:500]}...")

In [ ]:
# Cell 6 — Display extracted frames
import json
from IPython.display import display, Image
from pathlib import Path

with open('/content/result.json') as f:
    result = json.load(f)

print(f"{result.get('frame_count', 0)} frames extracted\n")
for i, desc in enumerate(result.get('keyframe_descriptions', [])):
    print(f"Frame {i}: {desc}")

# Show frame images if they still exist in /tmp
# (They are cleaned up after analysis — run extract-frames to keep them)
print("\nTip: to keep frames, run:")
print("  !python3 cli.py extract-frames '<url>' --output-dir /content/frames")

In [ ]:
# Cell 7 — Extract frames only (saves to /content/frames)
VIDEO_URL = 'https://www.youtube.com/watch?v=BaW_jenozKc'
!python3 cli.py extract-frames "{VIDEO_URL}" \
    --output-dir /content/frames \
    --interval 5 \
    --max-frames 10

from IPython.display import display, Image
import glob
for img_path in sorted(glob.glob('/content/frames/*.jpg'))[:6]:
    print(img_path)
    display(Image(img_path, width=400))

In [ ]:
# Cell 8 — Transcribe only
VIDEO_URL = 'https://www.youtube.com/watch?v=BaW_jenozKc'
!python3 cli.py transcribe "{VIDEO_URL}" \
    --model base \
    --output /content/transcript.json

import json
with open('/content/transcript.json') as f:
    t = json.load(f)
print(f"Language: {t['language']}")
print(f"\nFull transcript:\n{t['full_text']}")

In [ ]:
# Cell 9 — Full integration test WITH Claude vision (requires API key in cell 4)
!python3 tests/integration_test.py